In [1]:
# Step 1: Read the selected features from the output file generated in Part 2

selected_features_file = "output_ds.txt"

# Read the selected features from the file
with open(selected_features_file, "r") as file:
    selected_features = file.read().splitlines()

# Display the selected features
print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features[:10])  # Display the first 10 selected features


Number of selected features: 2000
Selected features: ['access', 'accessories', 'account', 'acid', 'acoustic', 'act', 'acted', 'acting', 'action', 'actions']


In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline

In [3]:
# Create a SparkSession
spark = SparkSession.builder \
    .appName("Text Classification with SVM") \
    .getOrCreate()

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/slf4j-log4j12-1.7.30.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.slf4j.impl.Log4jLoggerFactory]
24/05/26 20:52:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/05/26 20:52:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/05/26 20:52:04 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [4]:
# Suppress unnecessary logging
spark.sparkContext.setLogLevel("ERROR")

# If you're using Jupyter Notebook, you may want to display the output of each cell
# This command will make the output of the last line of each cell automatically displayed
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [5]:
# Step 2: Load the dataset and preprocess the data

# Load the dataset
data_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json" 

# Load the dataset into a Spark DataFrame
dataframe = spark.read.json(data_path)

# Preprocess the data
# Tokenization
tokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern="\s+|[,.;]", minTokenLength=2)
dataframe = tokenizer.transform(dataframe)

# Stopword removal
stop_words_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
dataframe = stop_words_remover.transform(dataframe)

# TF-IDF calculation
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=1000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[hashingTF, idf])
pipeline_model = pipeline.fit(dataframe)
dataframe = pipeline_model.transform(dataframe)

# Display the preprocessed dataframe schema and the first few rows
dataframe.printSchema()
dataframe.show(5)

root
 |-- asin: string (nullable = true)
 |-- category: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)

+----------+--------------------+-------+-------+--------------------+-----------+--------------+--------------------+-------------------+--------------+--------------------+--------------------+--------------------+--------------------+
|      asin|            category|helpful|ove

In [6]:
# Step 3: Split the dataset into training, validation, and test sets

# Define the proportions for splitting the dataset
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# Split the dataset into training, validation, and test sets
train_data, val_data, test_data = dataframe.randomSplit([train_ratio, val_ratio, test_ratio], seed=42)

# Cache the datasets for better performance
train_data.cache()
val_data.cache()
test_data.cache()

# Print the sizes of the datasets
print("Training set size:", train_data.count())
print("Validation set size:", val_data.count())
print("Test set size:", test_data.count())


DataFrame[asin: string, category: string, helpful: array<bigint>, overall: double, reviewText: string, reviewTime: string, reviewerID: string, reviewerName: string, summary: string, unixReviewTime: bigint, words: array<string>, filtered_words: array<string>, rawFeatures: vector, features: vector]

DataFrame[asin: string, category: string, helpful: array<bigint>, overall: double, reviewText: string, reviewTime: string, reviewerID: string, reviewerName: string, summary: string, unixReviewTime: bigint, words: array<string>, filtered_words: array<string>, rawFeatures: vector, features: vector]

DataFrame[asin: string, category: string, helpful: array<bigint>, overall: double, reviewText: string, reviewTime: string, reviewerID: string, reviewerName: string, summary: string, unixReviewTime: bigint, words: array<string>, filtered_words: array<string>, rawFeatures: vector, features: vector]

Training set size: 63165


Validation set size: 7775


Test set size: 7889


In [7]:
# Step 4: Apply vector length normalization using Normalizer with L2 norm

from pyspark.ml.feature import Normalizer

# Create a Normalizer transformer
normalizer = Normalizer(inputCol="features", outputCol="normalized_features", p=2.0)

# Apply normalization to training, validation, and test data
train_data = normalizer.transform(train_data)
val_data = normalizer.transform(val_data)
test_data = normalizer.transform(test_data)

# Display the first few rows of the normalized features in the training set
train_data.select("normalized_features").show(5, truncate=False)


+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|normalized_features                                                                                                                                                                                                                                                                                                                  

In [17]:
from pyspark.ml.classification import LinearSVC, OneVsRest
from pyspark.ml.feature import StringIndexer
from pyspark.ml.pipeline import Pipeline

# Create a StringIndexer transformer with a unique output column name
label_indexer = StringIndexer(inputCol="category", outputCol="label")

# Update the pipeline to include the StringIndexer
pipeline = Pipeline(stages=[hashingTF, idf, label_indexer, normalizer])

# Drop the existing "rawFeatures" column before applying the transformation
train_data = train_data.drop("rawFeatures", "features", "normalized_features")

# Fit the pipeline to the training data
pipeline_model = pipeline.fit(train_data)

# Transform the training data
train_data_transformed = pipeline_model.transform(train_data)

# Create a LinearSVC classifier
svm = LinearSVC(maxIter=10, regParam=0.1)

# Create an OneVsRest classifier
ovr = OneVsRest(classifier=svm)

# Train the OneVsRest classifier
ovr_model = ovr.fit(train_data_transformed)

# Get the underlying LinearSVC models
linear_svc_models = ovr_model.models

# Initialize a list to store the coefficients of each LinearSVC model
coefficients_list = []

# Loop through each LinearSVC model and get its coefficients
for idx, model in enumerate(linear_svc_models):
    coefficients_list.append((f"Coefficients for class {idx}", model.coefficients))

# Display the learned SVM model coefficients
for class_idx, coefficients in coefficients_list:
    print(class_idx, ":", coefficients)



Coefficients for class 0 : [0.01524198226550697,-0.04379509407208436,-0.01491194168499345,-0.06890813580023789,-0.02039756179708155,-0.012711312872475514,-0.011592391441824723,0.02594489395175088,0.02256052787330639,-0.006363007498506217,-0.005664001325839179,0.010295143994954434,0.00043657823900567584,-0.0037374737246352423,0.018082851135036285,-0.0008415350464131088,-0.016098089592734267,0.003001679231887557,-0.0169214360475379,-0.03324756317179656,-0.005837291704057253,-0.009073582168247978,-0.019372514856671026,0.017291177504046375,0.0007204176207357564,0.007972028711285896,-0.006046128781185486,0.006514672788508654,0.01937940933045334,-0.007130462075433265,-0.007280895036228069,0.013315183093566853,-0.010341770004726544,0.008527597036482936,-0.01943998847657246,-0.005231660694833607,-0.002546203278912995,-0.010456590329102403,0.00526774231505981,0.010331284154683379,0.010927215894735439,-0.0015010580460505778,-0.007456794054589684,-0.029209860472039356,0.004168206654220576,-0.0278

In [18]:
# Define the file path to save the coefficients
output_file_path = "svm_model_coefficients.txt"

# Open the file in write mode
with open(output_file_path, "w") as output_file:
    # Write the learned SVM model coefficients to the file
    for class_idx, coefficients in coefficients_list:
        output_file.write(f"{class_idx}: {coefficients}\n")

print("SVM model coefficients have been written to:", output_file_path)


20988

21269

21454

21155

21728

20484

... (13 outputs left out)

20625

21294

21803

SVM model coefficients have been written to: svm_model_coefficients.txt


In [19]:
spark.stop()